In [1]:
import requests
import datetime, time, logging, os, traceback
import pandas as pd

In [35]:
# today = datetime.date.today()
today = datetime.datetime(2026, 4, 15)
today_str = today.strftime('%d/%m/%Y')

In [36]:
df_separacoes_total = pd.DataFrame()

# logger.info("Buscando separacoes na página 1")
res_get_id = requests.get('https://api.tiny.com.br/api2/separacao.pesquisa.php', 
                params={'token': 'a841ba4682810a16845a10c47e790304f55f3fc5',
                        'formato': 'json',
                        'dataInicial': today_str,
                        'dataFinal': today_str
                        }
                )

separacoes = [obj['id'] for obj in res_get_id.json()['retorno']['separacoes']]

# logger.info(f"{len(separacoes)} separacoes encontradas na pagina 1")
df_separacoes = pd.DataFrame(separacoes, columns=['idSeparacao'])

df_separacoes_total = pd.concat([df_separacoes_total, df_separacoes], ignore_index=True)

page_qty = res_get_id.json()['retorno']['numero_paginas']

In [37]:
res_get_id.json()

{'retorno': {'status_processamento': '3',
  'status': 'OK',
  'separacoes': [{'id': '1075260780',
    'idOrigem': '1075260773',
    'objOrigem': 'notafiscal',
    'situacao': '3',
    'dataCriacao': '15/04/2026',
    'dataSeparacao': '15/04/2026',
    'dataCheckout': '15/04/2026',
    'destinatario': 'Willian Sanches',
    'idContato': '1031463277',
    'numero': '184266',
    'dataEmissao': '15/04/2026',
    'idFormaEnvio': '737931940',
    'numeroPedidoEcommerce': '260415T74DTBYN',
    'idOrigemVinc': '1075256260',
    'objOrigemVinc': 'venda',
    'situacaoVenda': '6'},
   {'id': '1075260797',
    'idOrigem': '1075260775',
    'objOrigem': 'notafiscal',
    'situacao': '3',
    'dataCriacao': '15/04/2026',
    'dataSeparacao': '15/04/2026',
    'dataCheckout': '15/04/2026',
    'destinatario': 'Wagner Santos de jesus',
    'idContato': '1031463283',
    'numero': '184267',
    'dataEmissao': '15/04/2026',
    'idFormaEnvio': '737931940',
    'numeroPedidoEcommerce': '260415T793JXUK'

In [4]:
for page in range(2, page_qty + 1):
        # logger.info(f"Buscando separacoes na pagina {page}")
        separacoes = []        
        df_separacoes = pd.DataFrame()

        res_get_id = requests.get('https://api.tiny.com.br/api2/separacao.pesquisa.php', 
                        params={'token': 'a841ba4682810a16845a10c47e790304f55f3fc5',
                                'formato': 'json',
                                'dataInicial': today_str,
                                'dataFinal': today_str,
                                'pagina': page
                                }
                        )
        
        separacoes = [obj['id'] for obj in res_get_id.json()['retorno']['separacoes']]
        
        # logger.info(f"{len(separacoes)} separacoes encontradas na pagina {page}")
        
        df_separacoes = pd.DataFrame(separacoes, columns=['idSeparacao'])

        df_separacoes_total = pd.concat([df_separacoes_total, df_separacoes], ignore_index=True)

        # logger.info(f'Encontradas {len(df_separacoes_total)} separacoes.')

In [28]:
res_get_id.json()['retorno']['separacoes']

[{'id': '1075304260',
  'idOrigem': '1075304239',
  'objOrigem': 'notafiscal',
  'situacao': '3',
  'dataCriacao': '17/04/2026',
  'dataSeparacao': '17/04/2026',
  'dataCheckout': '17/04/2026',
  'destinatario': 'Mauricio da Conceicao Oliveira da Silva',
  'idContato': '1031459606',
  'numero': '185550',
  'dataEmissao': '17/04/2026',
  'idFormaEnvio': '675234044',
  'numeroPedidoEcommerce': '2000015956561898',
  'idOrigemVinc': '1075212750',
  'objOrigemVinc': 'venda',
  'situacaoVenda': '6'},
 {'id': '1075304263',
  'idOrigem': '1075304241',
  'objOrigem': 'notafiscal',
  'situacao': '2',
  'dataCriacao': '17/04/2026',
  'dataSeparacao': '17/04/2026',
  'dataCheckout': None,
  'destinatario': 'Hivo Quirino de Almeida',
  'idContato': '1031452261',
  'numero': '185551',
  'dataEmissao': '17/04/2026',
  'idFormaEnvio': '675234044',
  'numeroPedidoEcommerce': '2000015905369794',
  'idOrigemVinc': '1075160402',
  'objOrigemVinc': 'venda',
  'situacaoVenda': '6'},
 {'id': '1075304264',
  

In [10]:
# logger.info("Buscando produtos para separacao...")
df_produtos = pd.DataFrame()
for i, row in df_separacoes_total.iterrows(): 
        res_get_sep = requests.get('https://api.tiny.com.br/api2/separacao.obter.php', 
                params={'token': 'a841ba4682810a16845a10c47e790304f55f3fc5',
                        'formato': 'json',
                        'idSeparacao': row['idSeparacao']
                        }
                )
        if res_get_sep.json()['retorno']['status'] != 'OK':
                # logger.info(f"Falha ao obter separacao {row['idSeparacao']}: {res_get_sep.json()['retorno']['status']}, possivelmente por sobrecarga na API.\nAguardando 90 segundos para tentar novamente...")
                time.sleep(90)  # Espera 90 segundos antes de tentar novamente
                
                res_get_sep = requests.get('https://api.tiny.com.br/api2/separacao.obter.php', 
                        params={'token': 'a841ba4682810a16845a10c47e790304f55f3fc5',
                                'formato': 'json',
                                'idSeparacao': row['idSeparacao']
                                }
                )
        
        df_produto = pd.DataFrame(
                [(obj['idProduto'], obj['quantidade']) for obj in res_get_sep.json()['retorno']['separacao']['itens']],
                columns=['idProduto', 'quantidadeParaSeparacao']
        )
        
        df_produto['idSeparacao'] = row['idSeparacao']

        df_produtos = pd.concat([df_produtos, df_produto], ignore_index=True)

In [14]:
res_get_sep.json()

{'retorno': {'status_processamento': '3',
  'status': 'OK',
  'separacao': {'id': '1075308254',
   'idOrigem': '1075308239',
   'objOrigem': 'notafiscal',
   'situacao': '1',
   'situacaoCheckout': '1',
   'dataCriacao': '17/04/2026',
   'itens': [{'idProduto': '939811440',
     'descricao': 'Lubrificante em spray 300 ml/200 g VONDER PLUS',
     'codigo': '5165200300',
     'quantidade': '1.0000',
     'unidade': 'pç',
     'localizacao': 'A0801',
     'infoAdicional': ''},
    {'idProduto': '1055347481',
     'descricao': 'Broca de aço rápido 5,00 mm DIN 338 HSS VONDER',
     'codigo': '5337005000',
     'quantidade': '1.0000',
     'unidade': 'pç',
     'localizacao': '',
     'infoAdicional': ''}],
   'qtdVolumes': '0',
   'numero': '185570',
   'dataEmissao': '17/04/2026',
   'idFormaEnvio': '0',
   'formaEnvio': None,
   'idContato': '1031450109',
   'destinatario': 'CLIMBER COMERCIO E SERVIÇOS',
   'situacaoOrigem': None}}}

In [6]:
df_separacoes_total

,idSeparacao
0,1075301700
1,1075301715
2,1075301713
3,1075301716
4,1075301736
...,...
413,1075306429
414,1075306462
415,1075306474
416,1075307908
